# LLM 9일차 (1) — 비동기 처리 기초 : 코루틴 · `async` / `await` · `asyncio.run` · `gather`

교재 `AI Agent` **p.120~124** (비동기 처리 — 개념 · 필요성 · 코루틴 · 실행 방법 2가지 · `main()` 권장 방법) 를 **시간을 재며** 확인한 노트입니다. LangGraph 에이전트를 비동기로 바꾸는 실습은 [`langgraph_async_multi_agent_rag.ipynb`](./langgraph_async_multi_agent_rag.ipynb) 에 있습니다.

```python
async def say_hello():          # 코루틴 함수
    return '안녕하세요'

async def main():
    result = await say_hello()  # 방법 1 : async 함수 안에서 await
    print(result)

asyncio.run(main())             # 방법 2 : .py 파일에서 딱 한 번   /   Jupyter 에서는 await main()
```

> **출력 읽는 법** — API 를 부르지 않는다. '오래 걸리는 I/O' 는 `asyncio.sleep` · `time.sleep` 으로 흉내 냈고 **모든 셀을 실제로 실행**했다. 걸린 시간은 실행할 때마다 조금씩 다르다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

## 0. 설치 · 실행 방법

`asyncio` 는 파이썬 표준 라이브러리라 **설치할 것이 없다**. 맨 위부터 모두 실행 (정리 환경 : Python 3.13 · ipykernel). Jupyter · VS Code 노트북은 셀에서 바로 `await` 를 쓸 수 있다.

In [1]:
import asyncio, time, sys, subprocess, tempfile, os

def stopwatch():
    start = time.perf_counter()
    return lambda: f'{time.perf_counter() - start:.2f}초'

## 1. 동기 vs 비동기 — 카페 비유 (교재 p.120~122)

커피 한 잔 = 기다리는 시간 0.5초(바리스타가 만드는 동안 손님은 할 일이 없다 = **I/O 대기**). 세 명이 주문한다.

In [2]:
def order_sync(name):
    time.sleep(0.5)                       # 커피가 나올 때까지 카운터 앞에서 '멈춰' 기다림 (Blocking)
    return f'{name} 커피'

lap = stopwatch()
print([order_sync(n) for n in ['A', 'B', 'C']], '→ 동기 :', lap())

['A 커피', 'B 커피', 'C 커피'] → 동기 : 1.50초


In [3]:
async def order_async(name):
    await asyncio.sleep(0.5)              # 진동벨을 받고 자리로 — 기다리는 동안 다른 일을 할 수 있다
    return f'{name} 커피'

# (가) await 를 하나씩 차례로
lap = stopwatch()
r = [await order_async(n) for n in ['A', 'B', 'C']]
print(r, '→ 비동기 · 차례로 await :', lap())

# (나) 세 주문을 '동시에' 걸어 두고 모두 기다리기
lap = stopwatch()
r = await asyncio.gather(*(order_async(n) for n in ['A', 'B', 'C']))
print(r, '→ 비동기 · gather :', lap())

['A 커피', 'B 커피', 'C 커피'] → 비동기 · 차례로 await : 1.50초


['A 커피', 'B 커피', 'C 커피'] → 비동기 · gather : 0.50초


| 방식 | 걸린 시간 | 이유 |
|---|---|---|
| 동기 `time.sleep` × 3 | 약 1.5초 | 하나가 끝나야 다음 |
| 비동기 · 차례로 `await` × 3 | **약 1.5초** | `await` 는 '끝날 때까지 이 코루틴은 기다린다' 는 뜻 — 하나씩 걸면 동기와 같다 |
| 비동기 · `asyncio.gather` | 약 0.5초 | 세 개를 동시에 걸어 두고 기다리는 시간이 겹친다 |

- 📝 교재 p.120 '기다리지 않고 다음 작업을 먼저 수행' 은 **기다리는 동안 할 다른 작업이 있을 때만** 이득이 된다. `async` · `await` 를 붙이기만 하면 빨라지는 것이 아니다 — 동시에 걸어 둘 일(여러 사용자 · 여러 검색 · 여러 LLM 호출)이 있어야 한다

## 2. 코루틴 — 부르면 '실행' 이 아니라 '예약표' 가 나온다 (교재 p.123)

In [4]:
async def say_hello():
    print('  (say_hello 실행 중)')
    return '안녕하세요'

coro = say_hello()                   # 호출만 — 아직 아무것도 출력되지 않는다
print('호출 결과 :', type(coro).__name__, coro)
result = await coro                  # await 해야 실제로 실행된다
print('await 결과 :', result)

호출 결과 : coroutine <coroutine object say_hello at 0x7f71455053c0>
  (say_hello 실행 중)
await 결과 : 안녕하세요


- 교재 p.123 '함수 호출 시 즉시 실행되지 않고 …' 그대로다. `say_hello()` 는 **코루틴 객체**를 만들 뿐이고, `await`(또는 `asyncio.run`)가 실제로 돌린다
- `await` 를 잊으면 결과 대신 `<coroutine object …>` 가 남고 'was never awaited' 경고가 난다 — 비동기 코드에서 가장 흔한 실수

## 3. 실행 방법 2가지 — `.py` 파일 vs Jupyter (교재 p.124)

In [5]:
async def main():
    result = await say_hello()          # 방법 1 : async 함수 안에서 await
    print(result)

# Jupyter 노트북 안에서 asyncio.run 을 부르면?
coro = main()
try:
    asyncio.run(coro)
except RuntimeError as e:
    coro.close()                        # 실행되지 못한 코루틴 정리 (경고 방지)
    print('RuntimeError :', e)

await main()                            # Jupyter 에서는 이렇게

RuntimeError : asyncio.run() cannot be called from a running event loop
  (say_hello 실행 중)
안녕하세요


In [6]:
# .py 파일로 저장해 터미널처럼 실행하면 asyncio.run(main()) 이 정상 동작한다
script = '''
import asyncio

async def say_hello():
    return "안녕하세요"

async def main():
    result = await say_hello()
    print(result)

asyncio.run(main())
'''
with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, 'hello_async.py')
    open(path, 'w', encoding='utf-8').write(script)
    out = subprocess.run([sys.executable, path], capture_output=True, text=True)
print('python hello_async.py →', out.stdout.strip())

python hello_async.py → 안녕하세요


- 📝 교재 p.124 · p.128 의 `asyncio.run(main())` 은 **`.py` 파일용**이다. Jupyter 는 이미 이벤트 루프가 돌고 있어 `asyncio.run() cannot be called from a running event loop` 오류가 나므로 **`await main()`** 을 쓴다 (필기 `LangGraph_실습6` 의 주석이 맞다)
- '`main()` 하나에 모아 `asyncio.run` 을 딱 한 번' (p.124 권장 방법) 은 이벤트 루프를 하나만 만들라는 뜻이다

## 4. 비동기 함수 안에서 하면 안 되는 것 — 막는(Blocking) 호출

In [7]:
async def order_blocking(name):
    time.sleep(0.5)                       # async 함수 안이지만 '멈추는' 동기 sleep
    return f'{name} 커피'

lap = stopwatch()
await asyncio.gather(*(order_blocking(n) for n in ['A', 'B', 'C']))
print('gather + time.sleep :', lap(), '← gather 를 써도 동시에 되지 않는다')

lap = stopwatch()
await asyncio.gather(*(asyncio.to_thread(order_sync, n) for n in ['A', 'B', 'C']))
print('gather + to_thread(동기 함수) :', lap(), '← 동기 함수는 스레드로 보내면 겹친다')

gather + time.sleep : 1.50초 ← gather 를 써도 동시에 되지 않는다


gather + to_thread(동기 함수) : 0.50초 ← 동기 함수는 스레드로 보내면 겹친다


- 이벤트 루프는 **한 줄기**라서, `async def` 안에서 `time.sleep` · `requests.get` · 동기 `llm.invoke` 처럼 **막는 호출**을 하면 그동안 다른 코루틴도 모두 멈춘다
- 그래서 교재 p.127 2단계가 `invoke` → **`await … ainvoke`** 로 바꾸라고 한다. 비동기 함수가 없는 동기 코드는 `asyncio.to_thread` 로 스레드에 넘긴다(LangChain 의 `ainvoke` 도 비동기 구현이 없는 부품은 이렇게 처리한다)

## 5. I/O 바운드 vs CPU 바운드 (교재 p.121)

In [8]:
def cpu_work(n=3_000_000):
    return sum(i * i for i in range(n))   # 계산만 — 기다리는 시간이 없다

async def cpu_async():
    return cpu_work()

cpu_work()                               # 처음 한 번은 준비 시간이 섞이므로 미리 실행
lap = stopwatch(); [cpu_work() for _ in range(3)]; t_sync = lap()
lap = stopwatch(); await asyncio.gather(*(cpu_async() for _ in range(3))); t_async = lap()
print(f'CPU 계산 3번 — 동기 {t_sync} · gather {t_async}  ← 비슷하다')

CPU 계산 3번 — 동기 0.45초 · gather 0.46초  ← 비슷하다


- 비동기는 **기다림(I/O)** 을 겹치는 기술이다. 계산(CPU)만 하는 일은 `gather` 로 묶어도 빨라지지 않는다. LLM 호출 · 웹 검색 · DB 조회처럼 **응답을 기다리는** 일이 많은 에이전트에 잘 맞는 이유(교재 p.121)

## 정리

| 할 일 | 코드 |
|---|---|
| 코루틴 함수 | `async def f(): …` — 부르면 코루틴 객체(예약표) |
| 실행 1 | `async` 함수 안에서 `result = await f()` |
| 실행 2 | `.py` 파일 : `asyncio.run(main())` 딱 한 번 / Jupyter : `await main()` |
| 동시에 | `await asyncio.gather(f(1), f(2), f(3))` — 차례로 `await` 하면 동기와 같은 시간 |
| 막는 호출 | `async def` 안에서 `time.sleep` · 동기 `invoke` 금지 → `await asyncio.sleep` · `await ainvoke` · `asyncio.to_thread` |